In [0]:
from databricks.sdk import AccountClient
from databricks.sdk.service.iam import ComplexValue

In [0]:
_ambiente = "PRD"

if 'prd' in _ambiente.lower():
    ambiente = _ambiente.lower()
elif 'qa' in _ambiente.lower():
    ambiente = _ambiente.lower()
else:
    ambiente = 'dev'

print("AMBIENTE: ",ambiente)

In [0]:
secret_dtb = dbutils.secrets.get(scope=f"aps-kv-ansedados-bs-{ambiente}", key=f"aps-adb-{ambiente}-secret")
tenant_dtb = dbutils.secrets.get(scope=f"aps-kv-ansedados-bs-{ambiente}", key=f"aps-adb-{ambiente}-tenant")
clientId_dtb = dbutils.secrets.get(scope=f"aps-kv-ansedados-bs-{ambiente}", key=f"aps-adb-{ambiente}-application")

a = AccountClient(account_id="b9cf75be-6372-404e-9bcb-af2be42c90ac",
                host='https://accounts.azuredatabricks.net',
                azure_tenant_id=tenant_dtb,
                azure_client_id=clientId_dtb,
                azure_client_secret=secret_dtb)


In [0]:
def get_primary_email(user_info):
    if user_info.emails:
        for email in user_info.emails:
            if email.primary:  # Verifica se é o e-mail principal
                return email.value  # Retorna o e-mail principal
        # Caso nenhum e-mail seja marcado como "primary", retorna o primeiro
        return user_info.emails[0].value
    return None  # Retorna None se o campo emails estiver vazio


def extract_group_data(groups):
    data = []
    for group in groups:
        group_name = group.display_name
        group_id = group.id
        if group.members:
            for member in group.members:

                try:
                    user_info = a.users.get(member.value)  # para poder adicionar o email da pessoa
                except:
                    user_info = None #se for sp ele retorna none a info do email

                # Obtém o e-mail do usuário, se disponível
                member_email = get_primary_email(user_info) if user_info else None

                data.append({
                    "group_name": group_name,
                    "group_id": group_id,
                    "member_name": member.display if member.display else None,
                    "member_id": member.value,
                    "member_email": member_email
                })
                
        else:
            # Caso o grupo não tenha membros, adiciona uma entrada sem membros
            data.append({
                "group_name": group_name,
                "group_id": group_id,
                "member_name": None,
                "member_id": None
            })
    return data

In [0]:
# extrai os dados dos grupos
groups_list = a.groups.list()

# transofrma os dados com e-mails
group_data = extract_group_data(groups_list)

# transofrma em dataframe
df_gruposAcc = spark.createDataFrame(group_data)

In [0]:
df_gruposAcc.write.mode("overwrite").saveAsTable(f"uc_saudepetro_{ambiente}.default.grupos_acesso_acc")